In [10]:
import numpy as np 
import rasterio
from rasterio.enums import Resampling
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path

Code below performs:
1. geometric alignment (reprojecting the 1km grid to 100m)
2. features for training
3. runs Random Forest Regressor
4. exports a standardized high-resolution GEOTIFF

In [11]:
def downscale_air_quality(coarse_aq_path, fine_covariate_path, output_path):
    """
    Downscales a 1km air quality raster to a 100m grid using a 100m covariate.
    """
    print("--- Phase 1: Reading High-Resolution 100m Geometry ---")
    with rasterio.open(fine_covariate_path) as fine_src:
        fine_meta = fine_src.meta.copy()
        fine_trans = fine_src.transform
        fine_w, fine_h = fine_src.width, fine_src.height
        fine_crs = fine_src.crs
        
        # Read the 100m predictor layer (e.g., NDVI map or Terrain)
        fine_covariate = fine_src.read(1)
        # Create a mask for valid land/science data points
        fine_mask = (fine_covariate != fine_src.nodata) & (~np.isnan(fine_covariate))

    print("--- Phase 2: Projecting Coarse 1km Data to Match Window Canvas ---")
    with rasterio.open(coarse_aq_path) as coarse_src:
        coarse_aq_nodata = coarse_src.nodata
        
        # Reproject the 1km dataset to the exact bounds and size of the 100m grid
        coarse_aq_matching_grid = np.empty((fine_h, fine_w), dtype="float32")
        rasterio.warp.reproject(
            source=rasterio.band(coarse_src, 1),
            destination=coarse_aq_matching_grid,
            src_transform=coarse_src.transform,
            src_crs=coarse_src.crs,
            dst_transform=fine_trans,
            dst_crs=fine_crs,
            resampling=Resampling.nearest # Maintains original 1km blocks for training
        )

    print("--- Phase 3: Building Training Vectors and Masking Missing Pixels ---")
    # Clean and isolate overlapping valid pixels (ignoring clouds/NoData bounds)
    train_mask = (
        fine_mask & 
        (coarse_aq_matching_grid != coarse_aq_nodata) & 
        (~np.isnan(coarse_aq_matching_grid))
    )
    
    # X = High-res environmental features, Y = Air Quality proxy value
    X_train = fine_covariate[train_mask].reshape(-1, 1)
    y_train = coarse_aq_matching_grid[train_mask]

    if len(X_train) == 0:
        raise ValueError("Error: No overlapping valid science pixels found. Check cloud masks.")

    print(f"Training Regressor on {len(X_train)} data cells...")
    
    print("--- Phase 4: Training Random Forest Model ---")
    rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    print("--- Phase 5: Generating Fine-Grained 100m Predictions ---")
    # Flatten the native 100m grid for inference
    X_predict = fine_covariate.flatten().reshape(-1, 1)
    fine_pred_flat = rf.predict(X_predict)
    fine_pred_grid = fine_pred_flat.reshape(fine_h, fine_w)
    
    # Re-apply structural NoData bounds so the output clips perfectly to land shapes
    fine_pred_grid[~fine_mask] = -9999.0

    print("--- Phase 6: Saving Downscaled 100m GeoTIFF ---")
    fine_meta.update({
        'driver': 'GTiff',
        'dtype': 'float32',
        'count': 1,
        'nodata': -9999.0
    })
    
    with rasterio.open(output_path, "w", **fine_meta) as dst:
        dst.write(fine_pred_grid.astype("float32"), 1)
        dst.update_tags(1, name="downscaled_aq", description="100m Downscaled Map Layer")
        
    print(f"Success! Saved high-resolution map to: {output_path}")

Program execution aka the main code

In [ ]:
# Update these paths to match your directory tree setup
INPUT_1KM_TIFF = Path("outputs/modis/MODIS_1km_sample.tif") 
INPUT_100M_COVARIATE = Path("outputs/ndvi/S2_NDVI_100m.tif")
OUTPUT_100M_TIFF = Path("outputs/modis/DOWNSCALED_100m_AQ.tif")

try:
    downscale_air_quality(INPUT_1KM_TIFF, INPUT_100M_COVARIATE, OUTPUT_100M_TIFF)
except Exception as e:
    print(f"Execution failed: {e}")

--- Phase 1: Reading High-Resolution 100m Geometry ---
--- Phase 2: Projecting Coarse 1km Data to Match Window Canvas ---
Execution failed: outputs\modis\MODIS_1km_sample.tif: No such file or directory
